# 🚀 Colab Ollama MCP — Max Sức Mạnh A100 (40GB VRAM + Hot-Swap qua 80GB System RAM)

> **Dành cho dự án phức tạp:** Khai thác tối đa sức mạnh phần cứng **Google Colab Pro NVIDIA A100 (40.00 GB GPU VRAM, 83.5 GB System RAM, 235 GB SSD NVMe)**. Cho phép nạp đồng thời bộ 4 mô hình cao cấp và **hot-swap tức thì trong 1–2 giây** thông qua cơ chế Linux OS Page Cache.

---

## 🧭 Hướng Dẫn Chọn Notebook Phù Hợp Cho Scope Của Bạn

| Tiêu chí | ⚡ `colab_ollama_mcp.ipynb` (Standard T4) | 🚀 `colab_ollama_a100_max.ipynb` (Notebook này) |
| :--- | :--- | :--- |
| **Phần cứng mục tiêu** | **Tesla T4 (16 GB VRAM, 12 GB RAM)** | **NVIDIA A100 (40 GB VRAM, 83.5 GB RAM)** |
| **Mức tiêu hao Colab Compute Units** | Cực thấp (~1.5–1.9 CU/giờ) | Trung bình–Cao (~5.4 CU/giờ) |
| **Mô hình cốt lõi** | `qwen2.5-coder:14b` ⭐ | **Bộ tứ 4-Model Suite (`32b` + `14b` + `vision`)** |
| **Phạm vi task tối ưu** | Daily coding, sửa bug cục bộ, viết unit test, CRUD API. | **Heavy Agentic Coding, thiết kế hệ thống lớn, refactor hàng chục files, root-cause analysis đa module, đọc sơ đồ UI/kiến trúc.** |

---

## ⚡ Cơ Chế Hot-Swap 1–2 Giây qua 80GB System RAM (Linux OS Page Cache)

Khi tải cả 4 model về ổ cứng Colab và nạp thử qua một lần, nhân Linux (Linux Kernel) sẽ tự động lưu trữ và giữ toàn bộ trọng số mô hình (~55–70 GB) trong **80 GB System RAM còn trống** dưới dạng **Page Cache**.
- Khi Antigravity IDE gọi `deepseek-r1:32b`, Ollama nạp model vào GPU trong ~1 giây.
- Khi chuyển sang `qwen2.5-coder:14b` để gõ code nhanh, Ollama hoán đổi model qua bus PCIe siêu tốc từ System RAM mà **không phải đọc lại từ ổ cứng**.

> [!NOTE]
> **🛡️ Bảo đảm an toàn 100% không bị OOM (Out-Of-Memory):**
> 1. **GPU VRAM (40GB):** Cấu hình `OLLAMA_MAX_LOADED_MODELS=1` đảm bảo tại mỗi thời điểm chỉ có 1 model lớn (~20GB VRAM) chiếm dụng GPU, để dư >20GB VRAM cho context dài (32K–64K tokens).
> 2. **System RAM (83.5GB):** Linux Page Cache là bộ nhớ *reclaimable* (có thể thu hồi bất cứ lúc nào), Linux OOM-killer sẽ không bao giờ kích hoạt.
> 3. **Ổ cứng (235GB):** Cả 4 model chỉ chiếm ~58 GB, còn trống hơn 170 GB SSD.

### Step 1: Xác Thực Phần Cứng NVIDIA A100 & 40GB VRAM
Chạy lệnh sau để xác nhận máy ảo Google Colab của bạn đã được gắn GPU **NVIDIA A100-SXM4-40GB**.

In [ ]:
# Step 1: Kiểm tra phần cứng NVIDIA A100 GPU và dung lượng VRAM thực tế
!nvidia-smi

### Step 2: Cấu Hình Lưu Trữ Mô Hình (Ổ Cứng Colab NVMe Siêu Tốc)
Mặc định sử dụng ổ cứng NVMe nội bộ của Colab (`/root/.ollama/models`) để đạt tốc độ tải mô hình tối đa (>600 MB/s).

In [ ]:
import os

# Trên A100, ưu tiên SSD nội bộ của Colab để nạp vào 80GB RAM cache với tốc độ cao nhất
USE_GDRIVE = False

if USE_GDRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        models_dir = '/content/drive/MyDrive/ollama_models'
        os.environ['OLLAMA_MODELS'] = models_dir
        os.makedirs(models_dir, exist_ok=True)
        print(f"✅ Đã kích hoạt Google Drive cache: {models_dir}")
    except Exception as e:
        print(f"⚡ Sử dụng Colab NVMe SSD nội bộ (/root/.ollama/models).")
else:
    print("⚡ Sử dụng Colab NVMe SSD nội bộ (/root/.ollama/models) — Tối ưu cho tốc độ Hot-Swap.")

### Step 3: Cài Đặt Ollama & Khởi Chạy Daemon Tối Ưu Cho A100 Hot-Swap
Thiết lập các biến môi trường đặc biệt:
- `OLLAMA_MAX_LOADED_MODELS=1`: Tự động trục xuất model cũ về System RAM cache khi chuyển model, ngăn chặn 100% rủi ro tràn VRAM 40GB.
- `OLLAMA_NUM_PARALLEL=4`: Tận dụng băng thông tính toán khủng của A100 để xử lý nhiều luồng song song.
- `OLLAMA_KEEP_ALIVE=24h`: Giữ các model đã nạp sẵn sàng 24/7.

In [ ]:
import os
import subprocess
import time
import urllib.request
from IPython.display import clear_output

# 1. Cài đặt zstd và bản phát hành Ollama Linux chính thức
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Cấu hình biến môi trường cao cấp cho A100 (40GB VRAM + 80GB RAM Cache)
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_KEEP_ALIVE"] = "24h"          # Giữ model trong VRAM / Cache liên tục
env["OLLAMA_MAX_LOADED_MODELS"] = "1"      # Đảm bảo VRAM chỉ gánh 1 model lớn tại 1 thời điểm (Chống OOM)
env["OLLAMA_NUM_PARALLEL"] = "4"          # 4 luồng song song cho các tác vụ MCP agentic

# 3. Khởi chạy Ollama ngầm
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Xóa log để giải phóng DOM
clear_output(wait=True)

# 4. Chờ daemon phản hồi
time.sleep(3)
try:
    with urllib.request.urlopen("http://localhost:11434/") as resp:
        print("🎉 Ollama A100 Daemon đã sẵn sàng:", resp.read().decode("utf-8"))
except Exception as e:
    print("⚠️ Không thể kết nối tới Ollama daemon:", e)

### Step 4: Tải Bộ Tứ 4-Model Suite Cho Dự Án Phức Tạp (Combo 2)
Tải trước 4 mô hình chuyên dụng cho từng giai đoạn phát triển phần mềm:
1. **`deepseek-r1:32b` (~20 GB):** Architect & Deep Reasoning — Dùng ở đầu task để phân tích yêu cầu, thiết kế kiến trúc, lên kế hoạch chi tiết.
2. **`qwen2.5-coder:32b` (~20 GB):** Heavy Agentic Coding & Multi-file — Model coding mạnh nhất, ngang ngửa Claude 3.5 Sonnet / GPT-4o.
3. **`qwen2.5-coder:14b` (~9.5 GB):** Fast Iteration & Quick Edits — Gõ code siêu tốc 45+ tok/s cho các hàm nhỏ, unit tests.
4. **`minicpm-v` (~5.5 GB):** UI/Diagram Multimodal — Phân tích ảnh chụp màn hình UI, sơ đồ kiến trúc Mermaid/Flowchart.

In [ ]:
from IPython.display import clear_output

# Danh sách 4 model trong Combo 2 cho A100
MODELS_TO_INSTALL = [
    ("qwen2.5-coder:14b", "⚡ Daily Driver & Fast Coding"),
    ("qwen2.5-coder:32b", "🧠 Heavy Agentic Coding & Architecture"),
    ("deepseek-r1:32b",   "🔍 Deep Reasoning, Bug Hunter & Planning"),
    ("minicpm-v",          "👁️ UI/Diagram & Screenshot Multimodal"),
]

print("⏳ Bắt đầu tải bộ 4 mô hình cao cấp cho A100 (khoảng 3–5 phút trên mạng Colab 500MB/s)...")
for tag, role in MODELS_TO_INSTALL:
    print(f"\n📦 Đang tải {tag} ({role})...")
    !ollama pull {tag}

clear_output(wait=True)
print("✅ Hoàn tất tải toàn bộ mô hình! Danh sách hiện có trong ổ cứng:")
!ollama list

### Step 5: Kích Hoạt Warm-up (Khởi Động Nạp Trọng Số Vào 80GB System RAM)
Chạy một truy vấn 1-token qua từng model để hệ điều hành Linux nạp toàn bộ trọng số vào **80 GB System RAM Page Cache**. Sau bước này, việc switch giữa các model trong Antigravity IDE chỉ mất **1–2 giây**!

In [ ]:
import json
import urllib.request
from IPython.display import clear_output

warmup_models = ["qwen2.5-coder:14b", "qwen2.5-coder:32b", "deepseek-r1:32b", "minicpm-v"]
print("🔥 Đang làm nóng (warm-up) các model để đưa vào 80GB System RAM Cache...")

for m in warmup_models:
    try:
        payload = json.dumps({"model": m, "prompt": "hi", "stream": False}).encode("utf-8")
        req = urllib.request.Request("http://localhost:11434/api/generate", data=payload, headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            print(f"  ✨ {m}: Đã nạp thành công vào bộ nhớ cache!")
    except Exception as e:
        print(f"  ⚠️ {m}: Bỏ qua warm-up ({e})")

print("\n🎉 TẤT CẢ MODEL ĐÃ SẴN SÀNG TRONG 80GB SYSTEM RAM! Hot-swap tức thì 1–2s.")

### Step 6: Mở Cloudflare Quick Tunnel & Kết Nối Với Antigravity IDE
Khởi tạo đường hầm HTTPS bảo mật ra ngoài Internet để Antigravity IDE có thể gọi qua MCP.

In [ ]:
import re
import subprocess
import time
from IPython.display import clear_output

# 1. Tải và cài đặt Cloudflare Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Khởi chạy đường hầm chuyển tiếp traffic về localhost:11434
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# 3. Trích xuất URL public *.trycloudflare.com
tunnel_url = None
print("⏳ Đang thiết lập Cloudflare Quick Tunnel...")
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.2)

clear_output(wait=True)

if tunnel_url:
    print("=" * 78)
    print("🎉 COLAB OLLAMA A100 PRO TUNNEL IS ONLINE!")
    print(f"🔗 PUBLIC ENDPOINT URL: {tunnel_url}")
    print("=" * 78)
    print("\n🚀 CÁCH 1 (Dán trực tiếp URL vào đoạn chat Antigravity IDE):\n")
    print(f"   {tunnel_url}")
    print("   (Hệ thống sẽ tự động switch endpoint và kết nối tức thì!)\n")
    print("⚡ CÁCH 2 (Chạy lệnh trong Terminal máy Mac):\n")
    print(f"   python3 colab_ollama_mcp/server.py --set-endpoint {tunnel_url}")
    print("\n" + "-" * 78)
else:
    print("❌ Không thể tạo đường hầm Cloudflare. Vui lòng chạy lại cell này.")

### Step 7: Bảng Theo Dõi Tài Nguyên Thời Gian Thực (A100 VRAM, 80GB RAM & Active Model)
Xem chi tiết mức tiêu hao **VRAM của GPU A100 (40GB)**, **System RAM (83.5GB)**, **SSD Disk** và **Model đang nạp thực tế trong GPU**.

In [ ]:
import json
import shutil
import subprocess
import time
import urllib.request
from IPython.display import clear_output

AUTO_REFRESH = False
REFRESH_SECONDS = 5

def make_bar(pct: float, width: int = 24) -> str:
    filled = max(0, min(width, int(round((pct / 100.0) * width))))
    return "█" * filled + "░" * (width - filled)

def print_a100_telemetry() -> None:
    print("=" * 78)
    print(f"📊 COLAB A100 PRO SERVER TELEMETRY — {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}")
    print("=" * 78)

    # 1. GPU VRAM & Load
    try:
        out = subprocess.check_output([
            "nvidia-smi",
            "--query-gpu=name,temperature.gpu,utilization.gpu,memory.used,memory.total",
            "--format=csv,noheader,nounits",
        ], text=True).strip()
        for line in out.splitlines():
            name, temp, util, mem_used, mem_total = [x.strip() for x in line.split(",")]
            used_gb = float(mem_used) / 1024.0
            total_gb = float(mem_total) / 1024.0
            vram_pct = (float(mem_used) / float(mem_total)) * 100.0 if float(mem_total) > 0 else 0.0
            util_pct = float(util)
            print(f"🎮 GPU Hardware : {name}  (🌡️ {temp}°C)")
            print(f"⚡ GPU Core Load: [{make_bar(util_pct)}] {util_pct:5.1f}%")
            print(f"🧠 GPU VRAM     : [{make_bar(vram_pct)}] {vram_pct:5.1f}%  ({used_gb:.2f} GB / {total_gb:.2f} GB)")
    except Exception as e:
        print(f"⚠️ GPU Status   : {e}")

    # 2. System RAM (83.5 GB)
    try:
        meminfo = {}
        with open("/proc/meminfo", "r") as f:
            for line in f:
                parts = line.split(":")
                if len(parts) == 2:
                    meminfo[parts[0].strip()] = int(parts[1].strip().split()[0])
        total_ram_gb = meminfo.get("MemTotal", 0) / (1024 ** 2)
        avail_ram_gb = meminfo.get("MemAvailable", 0) / (1024 ** 2)
        used_ram_gb = total_ram_gb - avail_ram_gb
        ram_pct = (used_ram_gb / total_ram_gb * 100.0) if total_ram_gb > 0 else 0.0
        cached_gb = meminfo.get("Cached", 0) / (1024 ** 2)
        print(f"💻 System RAM   : [{make_bar(ram_pct)}] {ram_pct:5.1f}%  ({used_ram_gb:.2f} GB / {total_ram_gb:.2f} GB)")
        print(f"🚀 OS Page Cache: {cached_gb:.2f} GB (Đang đệm sẵn các trọng số model cho Hot-Swap!)")
    except Exception as e:
        print(f"⚠️ System RAM   : {e}")

    # 3. SSD Disk
    try:
        total_d, used_d, _ = shutil.disk_usage("/")
        used_d_gb = used_d / (1024 ** 3)
        total_d_gb = total_d / (1024 ** 3)
        disk_pct = (used_d / total_d * 100.0) if total_d > 0 else 0.0
        print(f"💾 Colab Disk   : [{make_bar(disk_pct)}] {disk_pct:5.1f}%  ({used_d_gb:.1f} GB / {total_d_gb:.1f} GB)")
    except Exception as e:
        print(f"⚠️ Colab Disk   : {e}")

    # 4. Active Models Pinned in GPU
    print("-" * 78)
    try:
        with urllib.request.urlopen("http://localhost:11434/api/ps", timeout=5) as resp:
            ps_data = json.loads(resp.read().decode("utf-8"))
            running = ps_data.get("models", [])
            if running:
                for m in running:
                    m_name = m.get("name", "unknown")
                    vram_gb = m.get("size_vram", 0) / (1024 ** 3)
                    total_gb = m.get("size", 0) / (1024 ** 3)
                    gpu_offload = (vram_gb / total_gb * 100.0) if total_gb > 0 else 100.0
                    print(f"🔥 Đang chạy trong VRAM: {m_name} ({vram_gb:.2f} GB — {gpu_offload:.0f}% GPU offload)")
            else:
                print("💤 VRAM GPU trống (Model sẽ được nạp tức thì trong 1s khi gửi câu hỏi)")
    except Exception as e:
        print(f"🔴 Ollama Status: {e}")
    print("=" * 78)

try:
    while True:
        clear_output(wait=True)
        print_a100_telemetry()
        if not AUTO_REFRESH:
            break
        print(f"🔄 Tự động cập nhật mỗi {REFRESH_SECONDS}s... (Nhấn Stop ⏹️ để dừng)")
        time.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print("⏹️ Đã dừng bảng theo dõi tài nguyên.")